In [4]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Embedding network
# -------------------------
class EmbeddingNet(nn.Module):
    def __init__(self, input_dim=128, embedding_dim=32):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, embedding_dim)
        )

    def forward(self, x):
        z = self.fc(x)
        # normalize for stable distance geometry
        z = nn.functional.normalize(z, p=2, dim=1)
        return z


# -------------------------
# Siamese wrapper
# -------------------------
class SiameseNet(nn.Module):
    def __init__(self, embedding_net):
        super().__init__()
        self.embedding_net = embedding_net

    def forward(self, x1, x2):
        z1 = self.embedding_net(x1)
        z2 = self.embedding_net(x2)
        return z1, z2


# -------------------------
# Contrastive loss
# -------------------------
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, z1, z2, y):
        # y: 1 for similar, 0 for dissimilar
        d = torch.norm(z1 - z2, p=2, dim=1)
        loss_pos = y * d.pow(2)                              # similar -> small distance
        loss_neg = (1 - y) * torch.clamp(self.margin - d,    # dissimilar -> distance >= margin
                                         min=0).pow(2)
        loss = 0.5 * (loss_pos + loss_neg).mean()
        return loss


# -------------------------
# Synthetic data generator
# -------------------------
def make_batch(batch_size=64, input_dim=128, same_prob=0.5):
    """
    Create a batch of pairs:
      - with probability same_prob: x2 ~ x1 + small noise  (similar)
      - otherwise:                   x2 ~ independent      (dissimilar)
    """
    x1 = torch.randn(batch_size, input_dim, device=device)

    # Which are similar pairs?
    y = (torch.rand(batch_size, device=device) < same_prob).float()

    # For similar pairs: x2 = x1 + small noise
    noise = 0.1 * torch.randn_like(x1)
    x2_sim = x1 + noise

    # For dissimilar pairs: independent random
    x2_diff = torch.randn(batch_size, input_dim, device=device)

    # Mix using y
    x2 = y.view(-1, 1) * x2_sim + (1 - y).view(-1, 1) * x2_diff

    return x1, x2, y


# -------------------------
# Training + evaluation demo
# -------------------------
embedding_net = EmbeddingNet(input_dim=128, embedding_dim=32).to(device)
model = SiameseNet(embedding_net).to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_steps = 1000
print_every = 100

model.train()
for step in range(1, num_steps + 1):
    x1, x2, y = make_batch(batch_size=64)
    optimizer.zero_grad()
    z1, z2 = model(x1, x2)
    loss = criterion(z1, z2, y)
    loss.backward()
    optimizer.step()

    if step % print_every == 0:
        # quick evaluation on a fresh batch
        with torch.no_grad():
            x1_val, x2_val, y_val = make_batch(batch_size=256)
            z1_val, z2_val = model(x1_val, x2_val)
            d_val = torch.norm(z1_val - z2_val, dim=1)

            # Distances for similar vs dissimilar
            d_pos = d_val[y_val == 1]
            d_neg = d_val[y_val == 0]

            # Simple threshold-based accuracy
            # we say "similar" if distance < 0.5
            threshold = 0.5
            pred = (d_val < threshold).float()
            acc = (pred == y_val).float().mean().item()

        print(
            f"Step {step:4d} | "
            f"Loss: {loss.item():.3f} | "
            f"d_pos mean: {d_pos.mean().item():.3f} | "
            f"d_neg mean: {d_neg.mean().item():.3f} | "
            f"Acc@thr=0.5: {acc*100:5.1f}%"
        )


Step  100 | Loss: 0.002 | d_pos mean: 0.098 | d_neg mean: 1.146 | Acc@thr=0.5: 100.0%
Step  200 | Loss: 0.002 | d_pos mean: 0.093 | d_neg mean: 1.121 | Acc@thr=0.5: 100.0%
Step  300 | Loss: 0.003 | d_pos mean: 0.092 | d_neg mean: 1.140 | Acc@thr=0.5: 100.0%
Step  400 | Loss: 0.003 | d_pos mean: 0.096 | d_neg mean: 1.140 | Acc@thr=0.5: 100.0%
Step  500 | Loss: 0.002 | d_pos mean: 0.086 | d_neg mean: 1.121 | Acc@thr=0.5: 100.0%
Step  600 | Loss: 0.002 | d_pos mean: 0.090 | d_neg mean: 1.136 | Acc@thr=0.5: 100.0%
Step  700 | Loss: 0.003 | d_pos mean: 0.090 | d_neg mean: 1.135 | Acc@thr=0.5: 100.0%
Step  800 | Loss: 0.002 | d_pos mean: 0.093 | d_neg mean: 1.140 | Acc@thr=0.5: 100.0%
Step  900 | Loss: 0.002 | d_pos mean: 0.087 | d_neg mean: 1.123 | Acc@thr=0.5: 100.0%
Step 1000 | Loss: 0.003 | d_pos mean: 0.087 | d_neg mean: 1.107 | Acc@thr=0.5: 100.0%


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Embedding network
# -------------------------
class EmbeddingNet(nn.Module):
    def __init__(self, input_dim=16, embedding_dim=2):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.ReLU(),
            nn.Linear(32, embedding_dim)
        )

    def forward(self, x):
        z = self.fc(x)
        # normalize so distances are in a nice range
        z = nn.functional.normalize(z, p=2, dim=1)
        return z


class SiameseNet(nn.Module):
    def __init__(self, embedding_net):
        super().__init__()
        self.embedding_net = embedding_net

    def forward(self, x1, x2):
        z1 = self.embedding_net(x1)
        z2 = self.embedding_net(x2)
        return z1, z2


class ContrastiveLoss(nn.Module):
    def __init__(self, margin=1.0):
        super().__init__()
        self.margin = margin

    def forward(self, z1, z2, y):
        d = torch.norm(z1 - z2, dim=1)
        loss_pos = y * d.pow(2)                              # similar -> small
        loss_neg = (1 - y) * torch.clamp(self.margin - d,    # dissimilar -> >= margin
                                         min=0).pow(2)
        return 0.5 * (loss_pos + loss_neg).mean()


# -------------------------
# Synthetic data with hidden classes
# -------------------------
K = 4  # number of classes
input_dim = 16

# fixed class centers
class_centers = torch.randn(K, input_dim, device=device)

def sample_from_class(c, n):
    # each class is a Gaussian around its center
    center = class_centers[c]
    return center + 0.5 * torch.randn(n, input_dim, device=device)

def make_batch(batch_size=64, same_prob=0.5):
    """
    Similar pair: same class
    Dissimilar pair: different classes
    """
    y = (torch.rand(batch_size, device=device) < same_prob).float()

    x1 = torch.empty(batch_size, input_dim, device=device)
    x2 = torch.empty(batch_size, input_dim, device=device)

    for i in range(batch_size):
        if y[i] == 1:  # similar => same class
            c = torch.randint(0, K, (1,)).item()
            x1[i] = sample_from_class(c, 1)
            x2[i] = sample_from_class(c, 1)
        else:          # dissimilar => different classes
            c1 = torch.randint(0, K, (1,)).item()
            c2 = (c1 + torch.randint(1, K, (1,)).item()) % K
            x1[i] = sample_from_class(c1, 1)
            x2[i] = sample_from_class(c2, 1)

    return x1, x2, y


# -------------------------
# Evaluation helper (fixed threshold)
# -------------------------
def eval_with_threshold(model, threshold, batch_size=512):
    model.eval()
    with torch.no_grad():
        x1, x2, y = make_batch(batch_size=batch_size)
        z1, z2 = model(x1, x2)
        d = torch.norm(z1 - z2, dim=1)

        d_pos = d[y == 1]
        d_neg = d[y == 0]

        pred = (d < threshold).float()  # "similar" if distance < thr
        acc = (pred == y).float().mean().item()

    return {
        "d_pos_mean": d_pos.mean().item(),
        "d_neg_mean": d_neg.mean().item(),
        "acc": acc,
    }


# -------------------------
# Model, loss, optimizer
# -------------------------
embedding_net = EmbeddingNet(input_dim=input_dim, embedding_dim=2).to(device)
model = SiameseNet(embedding_net).to(device)
criterion = ContrastiveLoss(margin=1.0)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# -------------------------
# Choose a threshold from the untrained model (e.g., median distance)
# -------------------------
model.eval()
with torch.no_grad():
    x1_init, x2_init, y_init = make_batch(batch_size=512)
    z1_init, z2_init = model(x1_init, x2_init)
    d_init = torch.norm(z1_init - z2_init, dim=1)
    threshold = d_init.median().item()

print(f"Initial threshold (frozen for demo): {threshold:.3f}")

# Initial evaluation
stats = eval_with_threshold(model, threshold)
print(f"Before training: d_pos={stats['d_pos_mean']:.3f}, "
      f"d_neg={stats['d_neg_mean']:.3f}, acc={stats['acc']*100:.1f}%")

# -------------------------
# Training loop
# -------------------------
num_steps = 2000
print_every = 200

model.train()
for step in range(1, num_steps + 1):
    x1, x2, y = make_batch(batch_size=64)
    optimizer.zero_grad()
    z1, z2 = model(x1, x2)
    loss = criterion(z1, z2, y)
    loss.backward()
    optimizer.step()

    if step % print_every == 0:
        stats = eval_with_threshold(model, threshold)
        print(
            f"Step {step:4d} | "
            f"Loss={loss.item():.3f} | "
            f"d_pos={stats['d_pos_mean']:.3f} | "
            f"d_neg={stats['d_neg_mean']:.3f} | "
            f"Acc={stats['acc']*100:.1f}%"
        )


Initial threshold (frozen for demo): 0.666
Before training: d_pos=0.486, d_neg=1.383, acc=78.5%
Step  200 | Loss=0.004 | d_pos=0.127 | d_neg=1.458 | Acc=99.6%
Step  400 | Loss=0.002 | d_pos=0.073 | d_neg=1.495 | Acc=100.0%
Step  600 | Loss=0.002 | d_pos=0.054 | d_neg=1.473 | Acc=100.0%
Step  800 | Loss=0.001 | d_pos=0.040 | d_neg=1.500 | Acc=100.0%
Step 1000 | Loss=0.000 | d_pos=0.038 | d_neg=1.452 | Acc=100.0%
Step 1200 | Loss=0.000 | d_pos=0.032 | d_neg=1.471 | Acc=100.0%
Step 1400 | Loss=0.000 | d_pos=0.022 | d_neg=1.521 | Acc=100.0%
Step 1600 | Loss=0.000 | d_pos=0.018 | d_neg=1.485 | Acc=100.0%
Step 1800 | Loss=0.000 | d_pos=0.016 | d_neg=1.492 | Acc=100.0%
Step 2000 | Loss=0.000 | d_pos=0.015 | d_neg=1.477 | Acc=100.0%


**Best Practices**

- Use data augmentation (rotation, scaling, brightness) to increase the diversity of the training data and improve the generalization

- Tune hyperparameters (learning rate, margin, batch size) in the contrastive loss function can significantly affect the performance of the Siamese network

- Choose subnetwork based on task complexity. For simple tasks, a small CNN may be sufficient, while for more complex tasks, deeper architectures like ResNet or VGG can be used

- Balance positive & negative pairs

A note on positive & negative pairs:

Siamese networks do not learn categories like: "this is a cat" or "this is a dog."

Instead, they learn a similarity metric:

similar → small distance

dissimilar → large distance

This only works because we provide the network with many examples of:

- How similar things should look (positive pairs)

- How different things should look (negative pairs)

A positive pair consists of two inputs that should be considered similar.

Examples:

- Vision

- - Two images of the same person (face recognition)

- - Two photos of the same object (object verification)

- - Two satellite/drone images of the same location (GPS-denied navigation)

- - Two signatures written by the same person (signature verification)

- Text / Data

- - Two questions asking for the same thing (duplicate detection)

- - Two sentences expressing the same intent

In training: If the pair is positive, the label is usually: $y=1$ and the loss (contrastive or triplet) tries to pull their embeddings closer


A negative pair consists of two inputs that should be considered different.

Examples:

- Vision

- - Two images of different people

- - Two photos of different objects

- - Two map tiles showing different locations

- - A genuine signature vs a forgery

- Text / Data

- - Two unrelated sentences

- - Two different questions from StackOverflow

- - Two different products in a recommendation system

In training: If the pair is negative, the label is usually: $y=0$ and the loss tries to push their embeddings apart, at least by a margin.